# Qwen PRM Scoring Smoke Test

Smoke test for `Qwen2.5-Math-PRM-7B`. The PRM scores reasoning steps
at `<extra_0>` separator tokens via a dedicated reward head that emits
an (incorrect, correct) probability pair at each separator.

Uses the flamingo toy example from the model card; expected scores are
roughly `[1.0, 0.1904, 0.9766, 1.0]`.

Note: the model card specifies `bfloat16`, but the V100 (sm_70) has no
bf16 support, so we load `float16`. fp16 preserves step *rankings* but
can drift the absolute scores, so don't expect an exact match. See the
`bfloat16-vs-float16-on-v100` findings note.

In [1]:
import gc

import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

from notebook_utils import gpu_mem_used_gb

base_dir = "/groups/chichengz/tnn/datasets"
qwen_prm_dir = f"{base_dir}/Qwen2.5-Math-PRM-7B"

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).


In [ ]:
# System prompt per the Qwen2.5-Math-PRM-7B model card.
# Double backslash so "\boxed" is a literal backslash, not a
# "\b" backspace.
qwen_system_prompt = (
    "Please reason step by step, and put your final answer "
    "within \\boxed{}."
)

# Toy example from the Qwen2.5-Math-PRM-7B model card.
# Double-backslash the LaTeX (\\times, \\boxed) so it survives
# intact — a single "\t"/"\b" would become a tab/backspace
# control char and corrupt the input.
# Expected scores (bf16, model card): [1.0, 0.1904, 0.9766, 1.0].
problem = (
    "Sue lives in a fun neighborhood.  One weekend, the "
    "neighbors decided to play a prank on Sue.  On Friday "
    "morning, the neighbors placed 18 pink plastic flamingos "
    "out on Sue's front yard.  On Saturday morning, the "
    "neighbors took back one third of the flamingos, painted "
    "them white, and put these newly painted white flamingos "
    "back out on Sue's front yard.  Then, on Sunday morning, "
    "they added another 18 pink plastic flamingos to the "
    "collection. At noon on Sunday, how many more pink "
    "plastic flamingos were out than white plastic flamingos?"
)

reasoning_steps = [
    "To find out how many more pink plastic flamingos were "
    "out than white plastic flamingos at noon on Sunday, we "
    "can break down the problem into steps. First, on Friday, "
    "the neighbors start with 18 pink plastic flamingos.",

    "On Saturday, they take back one third of the flamingos. "
    "Since there were 18 flamingos, (1/3 \\times 18 = 6) "
    "flamingos are taken back. So, they have (18 - 6 = 12) "
    "flamingos left in their possession. Then, they paint "
    "these 6 flamingos white and put them back out on Sue's "
    "front yard. Now, Sue has the original 12 pink flamingos "
    "plus the 6 new white ones. Thus, by the end of Saturday, "
    "Sue has (12 + 6 = 18) pink flamingos and 6 white "
    "flamingos.",

    "On Sunday, the neighbors add another 18 pink plastic "
    "flamingos to Sue's front yard. By the end of Sunday "
    "morning, Sue has (18 + 18 = 36) pink flamingos and "
    "still 6 white flamingos.",

    "To find the difference, subtract the number of white "
    "flamingos from the number of pink flamingos: "
    "(36 - 6 = 30). Therefore, at noon on Sunday, there were "
    "30 more pink plastic flamingos out than white plastic "
    "flamingos. The answer is (\\boxed{30}).",
]

In [ ]:
def load_qwen_prm(
    model_dir: str,
    device_map: str = "cuda:0",
):
    # float16 for V100 (sm_70); model card says bfloat16
    # (Ampere+), so scores may differ slightly on Ampere GPUs.
    tokenizer = AutoTokenizer.from_pretrained(
        model_dir, trust_remote_code=True
    )
    model = AutoModel.from_pretrained(
        model_dir,
        device_map=device_map,
        dtype=torch.float16,
        trust_remote_code=True,
    ).eval()
    return model, tokenizer


qwen_model, qwen_tokenizer = load_qwen_prm(qwen_prm_dir)

print(f"model dtype: {next(qwen_model.parameters()).dtype}")
print(f"GPU memory : {gpu_mem_used_gb():.2f} GB")

In [ ]:
def score_qwen_prm(
    model,
    tokenizer,
    problem: str,
    steps: list[str],
    system: str,
    step_separator: str = "<extra_0>",
) -> list[float]:
    # step_separator is the reserved token Qwen2.5-Math-PRM-7B
    # uses to mark step boundaries; the PRM head emits an
    # (incorrect, correct) probability pair at each occurrence.
    # The trailing separator gives the last step its own score
    # position.
    assistant = step_separator.join(steps) + step_separator
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": problem},
        {"role": "assistant", "content": assistant},
    ]
    # add_generation_prompt=False: keep the assistant turn
    # ending at the steps as written instead of appending a
    # "your turn" cue.
    conversation = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False,
    )
    input_ids = tokenizer.encode(
        conversation, return_tensors="pt"
    ).to(model.device)

    # add_special_tokens=False: avoid prepending BOS so we get
    # just the single token id for the separator literal.
    sep_ids = tokenizer.encode(
        step_separator, add_special_tokens=False
    )
    if len(sep_ids) != 1:
        raise ValueError(f"Expected one separator token, got {sep_ids}")
    sep_positions = (input_ids[0] == sep_ids[0]).nonzero(as_tuple=True)[0]
    # Guard against tokenizer merges that change the separator
    # count.
    if sep_positions.numel() != len(steps):
        raise RuntimeError(
            f"Expected {len(steps)} separators, "
            f"found {sep_positions.numel()}"
        )

    with torch.no_grad():
        logits = model(input_ids=input_ids)[0]

    # PRM head emits 2 logits per token; index 1 is P(correct).
    probs = F.softmax(logits, dim=-1)
    return probs[0, sep_positions, 1].detach().cpu().float().tolist()

In [ ]:
qwen_scores = score_qwen_prm(
    qwen_model, qwen_tokenizer, problem, reasoning_steps,
    qwen_system_prompt,
)

for idx, (step, score) in enumerate(
    zip(reasoning_steps, qwen_scores), start=1
):
    print(f"step {idx}: P(correct) = {score:.4f}")
    print(step)
    print()

In [ ]:
del qwen_model, qwen_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory: {gpu_mem_used_gb():.2f} GB")